# Unit 6 / Chapter 6: Quantum Linear Algebra and the HHL Algorithm for AI

> **Main Learning Objective:** Understand why linear algebra sits at the heart of both classical AI and quantum computing, learn how quantum states and gates naturally represent vectors and matrices, and see how the HHL algorithm solves linear systems exponentially faster than classical solvers under the right conditions.

| Section | Topic | Why it matters for AI |
|---|---|---|
| 6.1 | Why linear algebra matters for AI | Regression, PCA, kernels, neural nets, and attention are all matrix math |
| 6.2 | Quantum states as vectors, gates as matrices | Quantum computers are natural linear algebra machines |
| 6.3 | The HHL algorithm | Solve Ax = b exponentially faster (in the right regime) |
| 6.4 | Quantum linear algebra in real ML | Quantum PCA, quantum regression, quantum kernels revisited |

By the end of this unit you should be able to:

1. Recognize the three or four linear algebra operations that dominate modern AI.
2. See a quantum state as a normalized complex vector and a gate as a unitary matrix.
3. Describe what the HHL algorithm does, at a high level, and what it does not do.
4. Name the assumptions that HHL needs to actually give a speedup.
5. Point to at least two concrete AI applications where quantum linear algebra could plausibly help.

---

## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
# Standard imports for the unit.
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random

np.random.seed(6); random.seed(6)
plt.rcParams['figure.dpi'] = 100
print("Imports done.")

---
## Course check-in

This logs that you started **Unit 6**. Enter the email you signed up with so we can track your progress.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 6
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 6.1: Why Linear Algebra Matters for AI

Almost every modern AI algorithm reduces, at some point, to a linear algebra operation. If you can speed those operations up, you can speed up training or inference for a huge slice of the field.

## Where linear algebra shows up in AI

| Technique | Underlying linear algebra |
|---|---|
| Linear regression | Solve Ax = b for x, or compute (X^T X)^{-1} X^T y |
| Principal Component Analysis (PCA) | Eigendecomposition of the covariance matrix |
| Singular Value Decomposition (SVD) | Diagonalize a rectangular matrix, used for LSA and recommendation systems |
| Support Vector Machines | Solve a quadratic program that involves kernel Gram matrices |
| Neural networks | Every layer is a matrix multiplication followed by a nonlinearity |
| Attention in transformers | Scaled dot product of query and key matrices |

The classical cost of many of these operations is O(N^3) or O(N^2) in the number of data points or features, which is fine for medium data but becomes painful at scale.

## Where quantum computers can plausibly help

Quantum states are *natively* complex vectors, and quantum gates are unitary matrices. That means a quantum computer with n qubits manipulates a 2^n dimensional vector every time it applies a gate. If you can encode your data and problem into that structure, you might be able to work exponentially faster in the dimension. The catch, of course, is that you can only read one sample out of the result, so this speedup only pays off when you need a summary of the answer, not the whole vector.

### Visualization: how classical cost scales

Below is a plot of the classical cost of solving Ax = b as N grows. Notice how quickly the O(N^3) curve for Gaussian elimination climbs.

In [ ]:
Ns = np.array([10, 30, 100, 300, 1000, 3000, 10000])
gauss = Ns**3.0
mvm   = Ns**2.0

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(Ns, gauss, marker='o', label='Gaussian elimination  O(N^3)', color='#5B2C91')
ax.loglog(Ns, mvm,   marker='o', label='Matrix vector multiply  O(N^2)', color='#2A9D8F')
ax.set_xlabel('problem size N (log scale)')
ax.set_ylabel('cost in arithmetic operations (log scale)')
ax.set_title('Classical solvers get expensive fast')
ax.grid(True, which='both', alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()
print("Every added zero on N adds roughly three zeros to Gaussian elimination cost.")

---
# Section 6.2: Quantum States as Vectors, Gates as Matrices

A quantum state on n qubits is a complex vector of length 2^n whose entries sum in magnitude squared to 1. A quantum gate on those qubits is a 2^n by 2^n unitary matrix.

So a quantum program is, at its core, just this:

$$ |\psi_{\text{out}}\rangle = U_k U_{k-1} \cdots U_2 U_1 |\psi_{\text{in}}\rangle. $$

That is exactly the same shape as a matrix multiplication chain in a neural network. The difference is that quantum "layers" are constrained to be unitary (norm preserving), and the vector length is 2^n rather than n.

## Quick facts you should be able to state on demand

| Concept | Classical | Quantum |
|---|---|---|
| Data unit | Real vector in R^n | Complex vector in C^(2^n) with unit norm |
| Operation | Any matrix, including nonlinear | Unitary matrix only |
| Inner product | Dot product of two vectors | Overlap of two states, computed by measurement |
| Cost of applying an operation | O(N^2) generally | O(poly(n)) gates for many structured operators |

### Encoding a classical vector into a quantum state

The most common way to store a real vector inside a quantum state is called *amplitude encoding*. Given a length 2^n vector x, we normalize it and let its entries become the amplitudes of an n qubit state:

$$ x \in \mathbb{R}^{2^n} \; \longrightarrow \; |x\rangle = \frac{1}{\|x\|} \sum_{i=0}^{2^n - 1} x_i |i\rangle. $$

Two features of this encoding are worth memorizing. First, you only need n qubits to hold a 2^n dimensional vector, which is exponentially compact. Second, you cannot read the whole vector back; a single measurement gives you one index i drawn from the distribution |x_i / ||x|| |^2.

In [ ]:
# Encode a small real vector into a quantum state by hand.
x = np.array([3.0, 1.0, -2.0, 4.0])
norm = np.linalg.norm(x)
psi = x / norm  # this is our amplitude encoded state

print(f"Classical vector x:     {x}")
print(f"Norm of x:              {norm:.3f}")
print(f"Encoded quantum state:  {psi.round(3)}")
print(f"Sum of |amp|^2:         {np.sum(np.abs(psi)**2):.3f}   (should be 1)")

# Sample from this state 4000 times to show what "reading" a quantum state means.
probs = np.abs(psi)**2
samples = np.random.choice(len(psi), size=4000, p=probs)
counts = np.bincount(samples, minlength=len(psi)) / 4000

fig, ax = plt.subplots(figsize=(6, 3))
labels = [f"|{i:02b}>" for i in range(len(psi))]
ax.bar([i-0.18 for i in range(len(psi))], probs,  width=0.35, label='true probabilities', color='#5B2C91')
ax.bar([i+0.18 for i in range(len(psi))], counts, width=0.35, label='4000 measurements',  color='#2A9D8F')
ax.set_xticks(range(len(psi))); ax.set_xticklabels(labels)
ax.set_ylabel('probability'); ax.legend()
ax.set_title('Amplitude encoded state: measurement histogram matches |amp|^2')
plt.tight_layout(); plt.show()

---
# Section 6.3: The HHL Algorithm

The Harrow, Hassidim, Lloyd (HHL) algorithm, published in 2009, solves the linear system

$$ A |x\rangle = |b\rangle $$

on a quantum computer. Under the right conditions it runs in time roughly

$$ O\Big( \log(N) \cdot s^2 \cdot \kappa^2 / \varepsilon \Big), $$

where N is the matrix size, s is its sparsity, kappa is its condition number, and epsilon is the target accuracy. The classical best known solver for a general system is O(N^3), so HHL is *exponentially* faster in N when the assumptions hold.

## The four assumptions HHL needs

For HHL to actually deliver a speedup, all of these have to be true:

1. **You can prepare |b> efficiently.** Encoding a general vector b into a quantum state can itself be expensive.
2. **A is sparse (or you can apply e^{i A t} efficiently).** HHL uses the matrix as a Hamiltonian and needs to simulate it in polynomial time.
3. **A has a well behaved condition number.** If kappa is huge, HHL is not fast.
4. **You only want a summary of x, not x itself.** You get |x> as a quantum state, and reading it out fully would cost O(N) samples.

If any of these fail, the "exponential speedup" quietly disappears. Being honest about this is a big part of understanding modern quantum ML.

## The three big steps inside HHL

1. **Phase estimation** on the operator e^{i A t} to load the eigenvalues of A into an ancilla register.
2. **Controlled rotation** that multiplies each amplitude by 1/eigenvalue.
3. **Uncomputation and measurement** that leaves you holding a state proportional to A^{-1} |b>.

You do not need to implement HHL from scratch to work with it; you need to know when it applies. Modern quantum linear algebra builds on the same ideas but avoids some of the strict HHL assumptions (via block encoding and quantum singular value transformation, QSVT).

### Activity 6.3: HHL on a tiny system by direct simulation

Full HHL is a very deep circuit. Simulating it end to end is beyond one notebook cell, but we can compare what HHL is *supposed* to give you against the classical answer for a small system. That is a useful sanity check.

Below we solve a 2 by 2 system Ax = b classically, then show the state proportional to A^{-1} b that HHL would prepare.

In [ ]:
A = np.array([[1.0, 0.5],
              [0.5, 1.0]])
b = np.array([1.0, 0.0])

# Classical solve
x_classical = np.linalg.solve(A, b)

# The quantum state HHL would prepare is proportional to A^{-1} b, normalized
x_encoded = x_classical / np.linalg.norm(x_classical)

print(f"A =\n{A}")
print(f"b = {b}")
print(f"Classical x = A^-1 b = {x_classical.round(4)}")
print(f"HHL output |x> (normalized amplitudes) = {x_encoded.round(4)}")

# Inspect the eigenvalues (relevant to HHL's assumptions)
eigvals, _ = np.linalg.eigh(A)
kappa = eigvals.max() / eigvals.min()
print(f"Eigenvalues of A: {eigvals.round(4)}")
print(f"Condition number kappa: {kappa:.3f}   (small kappa is good news for HHL)")

# Visualize the state HHL prepares
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([0, 1], np.abs(x_encoded)**2, color=['#5B2C91', '#2A9D8F'])
ax.set_xticks([0, 1]); ax.set_xticklabels(['|0>', '|1>'])
ax.set_ylabel('P(measure this outcome)')
ax.set_title(f"HHL output state, measurement probabilities")
plt.tight_layout(); plt.show()
print("If you measured the HHL output many times, this bar chart is what you would see.")

<details>
<summary>Reflection: what did the quantum computer give you?</summary>

HHL returned a *state* proportional to A^{-1} b. To read the whole vector A^{-1} b you would need many measurements, which erases the exponential speedup. HHL pays off when you only want a summary, such as an expectation value like <x| M |x> for some observable M, or a probability of a single outcome. That is a subtlety worth remembering: HHL is a subroutine, not a solver you use in isolation.
</details>

---
# Section 6.4: Quantum Linear Algebra in Real ML

The importance of HHL for AI is less about "solve this 10 billion by 10 billion system" and more about *unlocking quantum versions of common ML subroutines that use A^{-1}* somewhere inside them.

## Three concrete quantum ML routines built on HHL like ideas

| Routine | Classical version | Quantum version | Where it might help |
|---|---|---|---|
| Least squares regression | Solve X^T X w = X^T y | Quantum least squares (Wiebe et al., 2012) | Very high dimensional features |
| PCA | Diagonalize covariance | Quantum PCA (Lloyd, Mohseni, Rebentrost, 2013) | Extracting top eigenvectors of a huge density matrix |
| Support Vector Machine | Solve kernel quadratic program | Quantum SVM (Rebentrost, Mohseni, Lloyd, 2014) | Nonlinearly separable data with a good kernel |

Every one of these has an "if you can prepare the input and only need a summary" caveat. The productive framing is: these are candidates worth prototyping, not guaranteed wins.

## Activity 6.4: Compare classical vs "quantum encoded" PCA outputs

Below we run PCA on a small dataset the classical way, then show what the quantum encoding of the top principal component would look like as a state. This is a sanity check on the encoding, not a real quantum PCA implementation.

In [ ]:
# Make a small 2D dataset with a clear principal component direction
rng = np.random.default_rng(3)
n = 200
angle = 0.6
R = np.array([[np.cos(angle), -np.sin(angle)],
              [np.sin(angle),  np.cos(angle)]])
scale = np.array([[3.0, 0.0], [0.0, 0.5]])
data = (rng.standard_normal((n, 2)) @ scale) @ R.T

# Classical PCA: eigendecomposition of covariance
X = data - data.mean(axis=0)
C = (X.T @ X) / (n - 1)
eigvals, eigvecs = np.linalg.eigh(C)
top_dir = eigvecs[:, np.argmax(eigvals)]

# Encode the top principal component as an amplitude encoded state
top_dir = top_dir / np.linalg.norm(top_dir)
if top_dir[0] < 0: top_dir = -top_dir  # sign convention for display

print(f"Top principal component (classical): {top_dir.round(4)}")
print(f"Encoded as quantum amplitudes:       |{top_dir[0]:+.3f}> * |0> + |{top_dir[1]:+.3f}> * |1>")
print(f"Measurement probabilities:           P(|0>) = {top_dir[0]**2:.3f}, P(|1>) = {top_dir[1]**2:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(data[:,0], data[:,1], alpha=0.4, color='#5B2C91')
axes[0].arrow(0, 0, 3*top_dir[0], 3*top_dir[1], color='red', width=0.05, length_includes_head=True)
axes[0].set_title('Data + top principal component'); axes[0].axis('equal')
axes[0].set_xlabel('x0'); axes[0].set_ylabel('x1')

axes[1].bar(['P(|0>)','P(|1>)'], top_dir**2, color=['#5B2C91', '#2A9D8F'])
axes[1].set_ylim(0, 1)
axes[1].set_title('Amplitude encoding of the top PC as a 1 qubit state')
plt.tight_layout(); plt.show()

---
# End-of-Unit Quiz

## Section A: Multiple choice

**1. Which of the following is NOT primarily a linear algebra operation at its core?**

A) Principal Component Analysis
B) Solving a linear regression
C) Sorting a list of integers
D) Applying a neural network layer

<details><summary>Answer 1</summary>**C).** Sorting is a comparison based algorithm, not a linear algebra operation.</details>

---

**2. A quantum state on n qubits is described by:**

A) A complex vector of length n
B) A complex vector of length 2^n with unit norm
C) An n by n unitary matrix
D) A real vector of length 2^n

<details><summary>Answer 2</summary>**B).** Amplitudes are complex, there are 2^n of them, and the state has unit norm.</details>

---

**3. The HHL algorithm solves which type of problem?**

A) Sort a list of numbers
B) Find the shortest path in a graph
C) Solve A x = b for x
D) Train a large neural network

<details><summary>Answer 3</summary>**C).** HHL prepares a state proportional to A^{-1} b.</details>

---

**4. Which of the following is NOT one of HHL's assumptions?**

A) The vector b can be prepared efficiently as a quantum state
B) The matrix A is sparse or has efficient Hamiltonian simulation
C) The condition number of A is well behaved
D) The user needs to read out the full solution vector

<details><summary>Answer 4</summary>**D).** HHL only gives you a state; reading the whole vector out requires O(N) samples, which destroys the speedup. HHL is useful when you only need a summary.</details>

---

**5. Amplitude encoding stores a length 2^n classical vector using how many qubits?**

A) 2^n qubits
B) n qubits
C) log(n) qubits
D) One qubit

<details><summary>Answer 5</summary>**B) n qubits.** That is why the encoding is called exponentially compact.</details>

## Section B: Short answer

**6. In one sentence, state why linear algebra is such a natural target for quantum computers.**

<details><summary>Sample answer</summary>Quantum states are complex vectors and quantum gates are unitary matrices, so a quantum computer with n qubits manipulates a 2^n dimensional vector every time it applies a gate.</details>

---

**7. Give one concrete AI technique whose classical bottleneck is essentially a linear algebra operation.**

<details><summary>Sample answer</summary>Any of: linear regression (solve X^T X w = X^T y), PCA (eigendecomposition of covariance), SVMs (kernel Gram matrix), transformer attention (query key dot product), or any neural network layer (matrix multiplication).</details>

---

**8. Why can HHL give an exponential speedup in principle, but often not in practice?**

<details><summary>Sample answer</summary>Because the speedup only survives if you can prepare the input state efficiently, apply the matrix efficiently, kappa is small, and you only need a summary of the output. If any of those fails, the speedup is lost.</details>

---

**9. Explain in your own words why measuring a quantum state does not give you back the full amplitude vector.**

<details><summary>Sample answer</summary>Measurement returns a single basis state drawn from the distribution |amplitude|^2, not the amplitudes themselves. Recovering the full vector would require many measurements, which erases any exponential speedup.</details>

---

**10. (Bonus) Name one AI application from Units 3 to 5 that could plausibly use quantum linear algebra as a subroutine.**

<details><summary>Sample answer</summary>Any of: quantum kernel SVM for classification (Unit 3.4), quantum PCA as a preprocessing step, quantum least squares regression in high dimensions, or any variational algorithm whose cost gradient can be recast as a linear system.</details>

---

## Unit 6 Summary

| Concept | Key idea |
|---|---|
| Linear algebra in AI | Regression, PCA, SVMs, and neural nets are all matrix math at their core |
| Quantum states as vectors | n qubits hold a 2^n dimensional complex vector with unit norm |
| Quantum gates as matrices | Every gate is a 2^n by 2^n unitary; a program is a product of unitaries |
| Amplitude encoding | Store a length 2^n vector using only n qubits |
| HHL | Solve A x = b, exponentially fast in the ideal case, if the assumptions hold |
| The four HHL assumptions | Efficient |b>, efficient A, small kappa, summary output only |
| Quantum ML routines built on HHL ideas | Quantum least squares, quantum PCA, quantum SVM |

---
## End-of-unit submission

Fill in your quiz answers and run this cell to submit.

In [ ]:
quiz_answers = {
    "q1": "",
    "q2": "",
    "q3": "",
    "q4": "",
    "q5": "",
    "q6_short_answer": "Type your answer to question 6 here.",
    "q7_short_answer": "Type your answer to question 7 here.",
    "q8_short_answer": "Type your answer to question 8 here.",
    "q9_short_answer": "Type your answer to question 9 here.",
    "q10_bonus":       "Type your bonus answer here, or leave blank."
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 6!")
print("If this was your 6th completed unit and the certificate threshold is reached, watch your inbox.")